In [1]:
# ============================
# P-VecPick • FINAL APP CELL
# Single-cell, Voilà-ready
# ============================
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink, HTML

# ---- Config ----
CSV_PATH = "vector_data.csv"  # Change if your CSV lives elsewhere

# ---- Load Data ----
try:
    df = pd.read_csv(CSV_PATH)
except Exception as e:
    raise SystemExit(f"❌ Could not read '{CSV_PATH}'. Error: {e}\n"
                     f"Tip: Put vector_data.csv next to your notebook OR update CSV_PATH.")

# Normalize columns (safety if user tweaks CSV)
expected_cols = ["Vector","Type","Goal","System","Method","Promoter","Tag","Resistance","CopyNumber","Notes"]
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise SystemExit(f"❌ CSV is missing columns: {missing}\nExpected: {expected_cols}")

# ---- Helper functions ----
def clean(x):
    if pd.isna(x): return ""
    return str(x).strip().casefold()

# species/family sets for forgiving matching
DICOTS   = {"arabidopsis","tomato","nicotiana","cotton","dicot","withania","pelargonium"}
MONOCOTS = {"rice","maize","wheat","barley","monocot"}

def is_goal_match(row_val, sel_val):
    rv, sv = clean(row_val), clean(sel_val)
    return sv in ("", "any") or rv == sv

def is_method_match(row_val, sel_val):
    rv, sv = clean(row_val), clean(sel_val)
    return sv in ("", "any") or rv == sv or rv == "any"

def is_system_match(row_val, sel_val):
    rv, sv = clean(row_val), clean(sel_val)
    if sv in ("", "any") or rv in ("", "any"): 
        return True
    # If user picks species, allow family-level matches:
    if sv in DICOTS:   return (rv in DICOTS) or (rv == sv) or (rv == "dicot")
    if sv in MONOCOTS: return (rv in MONOCOTS) or (rv == sv) or (rv == "monocot")
    # If user picked family
    if sv == "dicot":   return rv in DICOTS or rv == "dicot"
    if sv == "monocot": return rv in MONOCOTS or rv == "monocot"
    # Exact species fallback
    return rv == sv

def is_text_match(text, query):
    if not query: return True
    return query.casefold() in (text or "").casefold()

def opt_match(row_val, sel_val):
    rv, sv = clean(row_val), clean(sel_val)
    return sv in ("", "any") or rv == sv or rv == "any"

def score_row(row, sel):
    """Simple ranking: exact goal(2), system(1), method(1), promoter(1), tag(1), resistance(1)"""
    s = 0
    if is_goal_match(row["Goal"], sel["Goal"]):           s += 2
    if is_system_match(row["System"], sel["System"]):     s += 1
    if is_method_match(row["Method"], sel["Method"]):     s += 1
    if opt_match(row["Promoter"], sel["Promoter"]):       s += 1
    if opt_match(row["Tag"], sel["Tag"]):                 s += 1
    if opt_match(row["Resistance"], sel["Resistance"]):   s += 1
    if opt_match(row["Type"], sel["Type"]):               s += 1
    # small bonus if keyword hits
    if sel["Keyword"]:
        if is_text_match(str(row["Vector"])+" "+str(row["Notes"]), sel["Keyword"]): s += 1
    return s

def unique_plus_any(series):
    vals = [v for v in sorted({str(x) for x in series.dropna().tolist()}) if v]
    return ["Any"] + vals

# ---- Widgets ----
style = {"description_width": "90px"}
layout = widgets.Layout(width="360px")

type_dd      = widgets.Dropdown(options=unique_plus_any(df["Type"]),      description="Type:",      style=style, layout=layout)
goal_dd      = widgets.Dropdown(options=unique_plus_any(df["Goal"]),      description="Goal:",      style=style, layout=layout)
system_dd    = widgets.Dropdown(options=unique_plus_any(df["System"]),    description="System:",    style=style, layout=layout)
method_dd    = widgets.Dropdown(options=unique_plus_any(df["Method"]),    description="Method:",    style=style, layout=layout)
promoter_dd  = widgets.Dropdown(options=unique_plus_any(df["Promoter"]),  description="Promoter:",  style=style, layout=layout)
tag_dd       = widgets.Dropdown(options=unique_plus_any(df["Tag"]),       description="Tag:",       style=style, layout=layout)
res_dd       = widgets.Dropdown(options=unique_plus_any(df["Resistance"]),description="Resistance:",style=style, layout=layout)

keyword_tb   = widgets.Text(placeholder="Optional: keyword (vector or notes)", description="Keyword:", style=style, layout=widgets.Layout(width="740px"))

suggest_btn  = widgets.Button(description="Suggest Vectors", button_style="primary", layout=widgets.Layout(width="200px"))
reset_btn    = widgets.Button(description="Reset", layout=widgets.Layout(width="120px"))
export_btn   = widgets.Button(description="Export CSV", layout=widgets.Layout(width="150px"))

sort_dd      = widgets.Dropdown(options=["Score (best first)", "Vector (A→Z)"], description="Sort:", style=style, layout=layout)

header_html = HTML("<h3 style='margin:8px 0 0 0;'>🧬 P-VecPick — Plant Vector Picker</h3>"
                   "<div style='color:#444;margin:2px 0 10px 0;'>Select filters and click <b>Suggest Vectors</b>. Use ‘Any’ to stay flexible.</div>")

out = widgets.Output()

# ---- Filter + Render ----
def current_selection():
    return {
        "Type": type_dd.value,
        "Goal": goal_dd.value,
        "System": system_dd.value,
        "Method": method_dd.value,
        "Promoter": promoter_dd.value,
        "Tag": tag_dd.value,
        "Resistance": res_dd.value,
        "Keyword": keyword_tb.value.strip()
    }

def filter_rows(sel):
    # Base boolean mask with core fields (for performance we combine later)
    mask = df.apply(lambda r:
                    is_goal_match(r["Goal"], sel["Goal"])
                    and is_system_match(r["System"], sel["System"])
                    and is_method_match(r["Method"], sel["Method"])
                    and opt_match(r["Promoter"], sel["Promoter"])
                    and opt_match(r["Tag"], sel["Tag"])
                    and opt_match(r["Resistance"], sel["Resistance"])
                    and opt_match(r["Type"], sel["Type"])
                    and is_text_match(f"{r['Vector']} {r['Notes']}", sel["Keyword"]),
                    axis=1)
    return df[mask].copy()

def run_search(_=None):
    clear_output(wait=True)
    display(header_html,
            widgets.HBox([type_dd, goal_dd, system_dd], layout=widgets.Layout(gap="10px")),
            widgets.HBox([method_dd, promoter_dd, tag_dd], layout=widgets.Layout(gap="10px")),
            widgets.HBox([res_dd, sort_dd], layout=widgets.Layout(gap="10px")),
            keyword_tb,
            widgets.HBox([suggest_btn, reset_btn, export_btn], layout=widgets.Layout(gap="10px", margin="6px 0")),
            out)
    sel = current_selection()

    with out:
        out.clear_output()
        hits = filter_rows(sel)

        if hits.empty:
            print("⚠️ No exact matches — showing closest suggestions instead.\n")
            scored = df.copy()
            scored["Score"] = scored.apply(lambda r: score_row(r, sel), axis=1)
            scored = scored.sort_values("Score", ascending=False)
            top = scored.head(10)[["Vector","Type","Goal","System","Method","Promoter","Tag","Resistance","Notes","Score"]]
            display(top.reset_index(drop=True))
            return

        # sort selection
        if "Score" in hits.columns:  # safety
            hits = hits.drop(columns=["Score"])
        if sort_dd.value.startswith("Score"):
            hits["Score"] = hits.apply(lambda r: score_row(r, sel), axis=1)
            hits = hits.sort_values("Score", ascending=False)
        else:
            hits = hits.sort_values("Vector", ascending=True)

        cols = ["Vector","Type","Goal","System","Method","Promoter","Tag","Resistance","CopyNumber","Notes"]
        print(f"🔍 Found {len(hits)} matching vectors:\n")
        display(hits[cols].reset_index(drop=True))

        # stash for export
        export_btn.hits_df = hits[cols].copy()

def reset_filters(_=None):
    type_dd.value     = "Any"
    goal_dd.value     = "Any"
    system_dd.value   = "Any"
    method_dd.value   = "Any"
    promoter_dd.value = "Any"
    tag_dd.value      = "Any"
    res_dd.value      = "Any"
    sort_dd.value     = "Score (best first)"
    keyword_tb.value  = ""
    run_search()

def export_csv(_=None):
    # Use a stable filename
    fname = "pvecpick_results.csv"
    if getattr(export_btn, "hits_df", None) is None or export_btn.hits_df.empty:
        with out:
            print("ℹ️ Nothing to export yet. Click ‘Suggest Vectors’ first.")
        return
    export_btn.hits_df.to_csv(fname, index=False)
    with out:
        print(f"✅ Exported results to {fname}")
        display(FileLink(fname))

# ---- Wire events ----
suggest_btn.on_click(run_search)
reset_btn.on_click(reset_filters)
export_btn.on_click(export_csv)

# ---- Initial render ----
run_search()


Text(value='', description='Keyword:', layout=Layout(width='740px'), placeholder='Optional: keyword (vector or…

Output()